# VNExpress Recommendation Benchmark

**Ablation study** comparing CF, CB, and Hybrid approaches.

In [ ]:
# Step 1: Clone latest code and setup
!git clone https://github.com/GadGadGad/DS300-Final-Project.git project 2>/dev/null || (cd project && git fetch origin && git reset --hard origin/main)
%cd project
!pip install -q torch torch_geometric pandas numpy scikit-learn tqdm sentence-transformers
print('✅ Setup complete!')

In [ ]:
# Step 2: Copy raw data from Kaggle dataset
!mkdir -p data/raw data/processed results/ablation
!cp -r /kaggle/input/vnexpress-data/* data/raw/ 2>/dev/null || echo 'No raw data found'
!ls data/raw/

In [ ]:
# Step 3: Copy pre-built graphs OR generate fresh
# (If graphs exist in dataset, use them; otherwise generate)
import os
graph_exists = os.path.exists('/kaggle/input/vnexpress-data/processed/strict_g1')
if graph_exists:
    print('Using pre-built graphs from dataset...')
    !cp -r /kaggle/input/vnexpress-data/processed/* data/processed/
else:
    print('Generating graphs from raw data...')
    !python src/data/convert_to_gnn.py --graph-type user-article --output data/processed/strict_g1 --min-user-interactions 2 --min-article-interactions 2
    !python src/data/convert_to_gnn.py --graph-type hetero --output data/processed/strict_g2 --min-user-interactions 2 --min-article-interactions 2
    !python src/data/convert_to_gnn.py --graph-type with-negatives --output data/processed/strict_g3 --min-user-interactions 2 --min-article-interactions 2
print('✅ Data ready!')

In [ ]:
# Step 4: Run ablation study
!chmod +x scripts/run_ablation.sh
!./scripts/run_ablation.sh 2>&1 | tee ablation_log.txt

In [ ]:
# Step 5: Load and analyze results
import os, json, pandas as pd
results = []
for f in os.listdir('results/ablation'):
    if f.endswith('.json'):
        with open(f'results/ablation/{f}') as file:
            data = json.load(file)
            parts = f.replace('.json','').split('_')
            results.append({
                'Model': parts[1] if len(parts)>1 else 'unknown',
                'Graph': parts[2] if len(parts)>2 else 'unknown',
                'Protocol': parts[3] if len(parts)>3 else 'unknown',
                'Recall@10': data.get('recall@10', 0),
                'NDCG@10': data.get('ndcg@10', 0)
            })
df = pd.DataFrame(results)
print(f'Loaded {len(df)} results')
df.sort_values('Recall@10', ascending=False).head(15)

In [ ]:
# Step 6: Visualize
import matplotlib.pyplot as plt
if len(df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    df_sorted = df.sort_values('Recall@10', ascending=True).tail(15)
    ax.barh(df_sorted['Model'] + '_' + df_sorted['Graph'], df_sorted['Recall@10'])
    ax.set_xlabel('Recall@10')
    ax.set_title('Top 15 Model Configurations')
    plt.tight_layout()
    plt.savefig('results/model_comparison.png', dpi=150)
    plt.show()
else:
    print('No results to plot')

In [ ]:
# Save summary
if len(df) > 0:
    df.to_csv('results/ablation_summary.csv', index=False)
    print('Summary saved to results/ablation_summary.csv')